In [1]:
import xarray as xr
import numpy as np
import dask

In [ ]:
path = '/jetfs/scratch/psplechtna/thesisdata/sfcfluxes/'
years = np.arange(1991, 2021, 1)

In [39]:
all_fluxes = xr.open_mfdataset([path+f'fluxes_per_sec_{year}.nc' for year in years]).sel(latitude=slice(90,0)).drop_vars(['number', 'expver'])
all_fluxes = all_fluxes.chunk({"valid_time": -1, "latitude": 20, "longitude": 120})
all_fluxes = all_fluxes.sel(valid_time=all_fluxes.valid_time.dt.month.isin([5, 6, 7, 8]))

In [41]:
t = all_fluxes.valid_time.dt
all_fluxes = all_fluxes.assign_coords(mdh=("valid_time", (t.month * 10000 + t.day * 100 + t.hour).data))

clim = all_fluxes[["slhf", "sshf"]].groupby("mdh").mean("valid_time")

In [42]:
clim

<xarray.Dataset> Size: 12GB
Dimensions:    (mdh: 2952, latitude: 361, longitude: 1440)
Coordinates:
  * mdh        (mdh) int64 24kB 50100 50101 50102 50103 ... 83121 83122 83123
  * latitude   (latitude) float64 3kB 90.0 89.75 89.5 89.25 ... 0.5 0.25 0.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    slhf       (mdh, latitude, longitude) float32 6GB dask.array<chunksize=(1, 20, 120), meta=np.ndarray>
    sshf       (mdh, latitude, longitude) float32 6GB dask.array<chunksize=(1, 20, 120), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-17T07:22 GRIB to CDM+CF via cfgrib-0.9.1...

In [43]:
counts = all_fluxes.valid_time.groupby(all_fluxes.mdh).count()
print(counts.min().item(), counts.max().item())

30 30


In [ ]:
clim.to_zarr(path + 'clim_fluxes.zarr')

/jetfs/home/psplechtna/.conda/envs/flexpart_post/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/jetfs/home/psplechtna/.conda/envs/flexpart_post/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2021312/1797421356.py", line 1, in <module>
    clim.to_zarr(out_dir + 'clim_fluxes.zarr')
  File "/jetfs/home/psplechtna/.conda/envs/flexpart_post/lib/python3.11/site-packages/xarray/core/dataset.py", line 2468, in to_zarr
    return to_zarr(  # type: ignore[call-overload,misc]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/jetfs/home/psplechtna/.conda/envs/flexpart_post/lib/python3.11/site-packages/xarray/backends/writers.py", line 782, in to_zarr
    writes = writer.sync(
             ^^^^^^^^^^^^
  File "/jetfs/home/psplechtna/.conda/envs/flexpart_post/lib/python3.11/site-packages/xarray/backends/common.py", line 420, in sync
    delayed_store = chunkmanager.store(
                    ^^^^^^^^^^^^^^^^^^^
  File "